# Handling missing categorical data
Rebuilt for pandas 3 and scikit-learn 1.9. Data: `data/house_prices.csv`, five columns of the training file of Kaggle's *House Prices: Advanced Regression Techniques* (1,460 houses, 81 columns in full).
Every imputer is fitted on the training set only.

In [ ]:
# pandas for the table, numpy for numbers, Plotly for charts, scipy for the KDE curves
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

## 1. The data and its gaps

In [ ]:
# load the two categorical columns and the target (the last two columns are for Section 6)
df = pd.read_csv("data/house_prices.csv")
print(df.shape)
df.head()

In [ ]:
# share of missing values per column, in %
(df.isnull().mean() * 100).round(2)

## 2. Split first, so the imputers learn from the training set only

In [ ]:
# inputs: the two categorical columns; output: the sale price
X = df[["GarageQual", "FireplaceQu"]]
y = df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)
X_train.isnull().sum()

## 3. Most frequent value (mode) imputation

In [ ]:
# the most frequent category of each column, from the training set
X_train.mode().iloc[0]

In [ ]:
# a helper: one KDE curve of sale prices, as a Plotly line
grid = np.linspace(0, 500_000, 400)
def kde_line(prices, name, dash="solid"):
    return go.Scatter(x=grid, y=gaussian_kde(prices)(grid), mode="lines", name=name, line=dict(dash=dash))

In [ ]:
# do houses with a gap sell like houses in the most frequent category?
fig = make_subplots(1, 2, subplot_titles=["GarageQual", "FireplaceQu"])
for j, (c, m) in enumerate([("GarageQual", "TA"), ("FireplaceQu", "Gd")], start=1):
    fig.add_trace(kde_line(y_train[X_train[c] == m], f"{c} = {m}"), 1, j)
    fig.add_trace(kde_line(y_train[X_train[c].isnull()], f"{c} = NaN"), 1, j)
fig.update_layout(template="simple_white", height=400)
fig.show()

In [ ]:
# fit the imputer on the training set; set_output keeps the column names
mode_imp = SimpleImputer(strategy="most_frequent").set_output(transform="pandas")
mode_imp.fit(X_train)
mode_imp.statistics_

In [ ]:
# fill both sets with what was learned from the training set
X_train_mode = mode_imp.transform(X_train)
X_test_mode = mode_imp.transform(X_test)
X_train_mode.isnull().sum().sum(), X_test_mode.isnull().sum().sum()

In [ ]:
# category shares (%) before and after, side by side
for c in ["GarageQual", "FireplaceQu"]:
    shares = pd.concat([X_train[c].value_counts(normalize=True),
                        X_train_mode[c].value_counts(normalize=True)], axis=1) * 100
    shares.columns = ["before", "after"]
    print(shares.round(1), "\n")

In [ ]:
# sale prices of the most frequent category, before and after filling the gaps
fig = make_subplots(1, 2, subplot_titles=["GarageQual = TA", "FireplaceQu = Gd"])
for j, (c, m) in enumerate([("GarageQual", "TA"), ("FireplaceQu", "Gd")], start=1):
    fig.add_trace(kde_line(y_train[X_train[c] == m], "before"), 1, j)
    fig.add_trace(kde_line(y_train[X_train_mode[c] == m], "after", "dash"), 1, j)
fig.update_layout(template="simple_white", height=400)
fig.show()

In [ ]:
# mean sale price of the most frequent category, before and after
for c, m in [("GarageQual", "TA"), ("FireplaceQu", "Gd")]:
    print(c, round(y_train[X_train[c] == m].mean()), round(y_train[X_train_mode[c] == m].mean()))

## 4. Missing category imputation

In [ ]:
# the constant strategy writes the same word into every gap
miss_imp = SimpleImputer(strategy="constant", fill_value="Missing").set_output(transform="pandas")
X_train_miss = miss_imp.fit_transform(X_train)
X_test_miss = miss_imp.transform(X_test)
miss_imp.statistics_

In [ ]:
# counts per category: 'Missing' is now a category of its own
for c in ["GarageQual", "FireplaceQu"]:
    print(X_train_miss[c].value_counts(), "\n")

## 5. The same with pandas (careful: learn the mode from the training set)

In [ ]:
# pandas way: compute the mode on the training set, then fill both sets with it
garage_mode = X_train["GarageQual"].mode()[0]
pd_train = X_train["GarageQual"].fillna(garage_mode)
pd_test = X_test["GarageQual"].fillna(garage_mode)
# same result as SimpleImputer
(pd_train == X_train_mode["GarageQual"]).all(), (pd_test == X_test_mode["GarageQual"]).all()

## 6. What the gaps really mean in this data

In [ ]:
# every house with no fireplace has FireplaceQu missing, and every house with no garage has GarageQual missing
print((df["FireplaceQu"].isnull() == (df["Fireplaces"] == 0)).all())
print((df["GarageQual"].isnull() == df["GarageType"].isnull()).all())